# Coral Reef Dataset Merge Pipeline (v2 — classification + segmentation)

Fusionne 5 datasets Roboflow (coraux) en un dataset unifié utilisable à la
fois pour la **classification** (Healthy / Dead_Coral / Bleached_Coral) et
la **segmentation sémantique**.

## Corrections par rapport à la v1

1. **Déduplication globale** (§ problème 1) : la v1 copiait le split
   `train`/`valid`/`test` de chaque source directement dans le dataset
   fusionné, sans dédupliquer entre sources — 18% du test et 14% du valid
   se sont révélés être des doublons exacts d'images du train (fuite de
   données, vérifié par hash SHA256). Corrigé : déduplication sur
   l'ensemble du dataset fusionné, tous splits confondus, APRÈS le merge
   mais AVANT l'augmentation.

2. **Masques de segmentation récupérés** (§ problème 2) : 4 des 5 sources
   (`coral_bleaching_fish_reefs`, `graduation_design`, `marjan_balance`,
   `surge`) sont au format **COCO avec polygones de segmentation par
   instance** — la v1 utilisait ces polygones uniquement pour **recadrer**
   l'image en patch de classification, en jetant l'information pixel par
   pixel. Corrigé : les polygones sont maintenant **rasterisés en masques
   de segmentation réels** (image complète, pas de recadrage), et
   conservés en plus du label de classification.
   Seule `bleach_detection` (type `folder`, classification pure, pas de
   polygones) reste sans masque — ces échantillons sont marqués
   `has_mask=0` dans le manifeste plutôt que de recevoir un faux masque.

## Structure de sortie

```
merged_dataset/
├── train/
│   ├── images/*.jpg          # image complète (pas de recadrage)
│   ├── masks/*.png           # masque pixel-classe (0=fond,1=Healthy,2=Dead_Coral,3=Bleached_Coral)
│   │                         #   -- uniquement pour les échantillons has_mask=1
│   └── labels.csv            # filename,class_id,has_mask
├── valid/  (même structure)
└── test/   (même structure)
```

Compatible avec `datasets/coral_dataset.py::CoralDataset` du pipeline
UWOC-SemCom (images/, masks/, labels.csv) -- ajouter la colonne `has_mask`
au chargement pour ignorer la perte de segmentation sur les échantillons
sans masque réel (voir note en fin de notebook).

## Step 0 — Configuration et dépendances

In [1]:
!pip install numpy Pillow pandas tqdm

import os, json, glob, shutil, random, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image, ImageDraw
import pandas as pd
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

# ---- EDIT THESE PATHS to match your workspace root ----
ROOT = Path(".")   # run notebook from the folder containing the 5 dataset folders

DATASETS = {
    "bleach_detection": {
        "path": ROOT / "Coral Reef Bleach Detection.v2i.folder",
        "type": "folder",
    },
    "coral_bleaching_fish_reefs": {
        "path": ROOT / "Coral Reef Bleaching.v17-fish-reefs.coco",
        "type": "coco",
    },
    "graduation_design": {
        "path": ROOT / "Graduation Design.v1i.coco",
        "type": "coco",
    },
    "marjan_balance": {
        "path": ROOT / "marjan balance.v5i.coco",
        "type": "coco",
    },
    "surge": {
        "path": ROOT / "surge.v1-2026-07-30-11-30.coco",
        "type": "coco",
    },
}

MERGED_DIR = ROOT / "merged_dataset"
CLASSES = ["Healthy", "Dead_Coral", "Bleached_Coral"]
CLASS_TO_ID = {c: i + 1 for i, c in enumerate(CLASSES)}   # 0 réservé au fond dans les masques
SPLITS = ["train", "valid", "test"]

for split in SPLITS:
    (MERGED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / "masks").mkdir(parents=True, exist_ok=True)

print("Datasets configurés :")
for name, cfg in DATASETS.items():
    exists = cfg["path"].exists()
    print(f"  {name:30s} type={cfg['type']:6s} exists={exists}  path={cfg['path']}")


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Datasets configurés :
  bleach_detection               type=folder exists=True  path=Coral Reef Bleach Detection.v2i.folder
  coral_bleaching_fish_reefs     type=coco   exists=True  path=Coral Reef Bleaching.v17-fish-reefs.coco
  graduation_design              type=coco   exists=True  path=Graduation Design.v1i.coco
  marjan_balance                 type=coco   exists=True  path=marjan balance.v5i.coco
  surge                          type=coco   exists=True  path=surge.v1-2026-07-30-11-30.coco


## Step 1 — Fusion de classes (`CLASS_MAP`)

Chaque source utilise des noms de catégories bruts différents. `CLASS_MAP`
les normalise vers les 3 classes unifiées. Complétez cette table si
`unmapped_classes_seen` (affiché à la fin du merge) contient des catégories
non reconnues propres à vos jeux de données.

In [2]:
CLASS_MAP = {
    # Healthy
    "healthy": "Healthy", "healthy coral": "Healthy", "Healthy": "Healthy",
    "Healthy_Coral": "Healthy", "healthy_coral": "Healthy", "live coral": "Healthy",
    "live_coral": "Healthy", "coral": "Healthy",
    # Dead
    "dead": "Dead_Coral", "dead coral": "Dead_Coral", "Dead_Coral": "Dead_Coral",
    "dead_coral": "Dead_Coral", "Dead Coral": "Dead_Coral",
    # Bleached
    "bleached": "Bleached_Coral", "bleached coral": "Bleached_Coral",
    "Bleached_Coral": "Bleached_Coral", "bleached_coral": "Bleached_Coral",
    "Bleached Coral": "Bleached_Coral", "bleaching": "Bleached_Coral",
}

unmapped_classes_seen = set()

def map_class(raw_name):
    if raw_name is None:
        return None
    if raw_name in CLASS_MAP:
        return CLASS_MAP[raw_name]
    # essai insensible à la casse / underscores-espaces
    normalized = raw_name.strip().lower().replace("_", " ")
    for k, v in CLASS_MAP.items():
        if k.strip().lower().replace("_", " ") == normalized:
            return v
    return None

## Step 2 — Ingestion : dataset `folder` (classification pure, pas de masque)

Copie l'image complète, écrit une ligne de manifeste avec `has_mask=0`.
Aucun recadrage : contrairement à la v1, on garde l'image entière (plus
cohérent pour un futur usage segmentation/detection, et n'affecte pas la
classification).

In [3]:
manifest_rows = []   # accumulé au fil du merge : dict(split, filename, class_id, has_mask, source)

def ingest_folder_dataset(name, path):
    counts = {"train": {}, "valid": {}, "test": {}}
    for split in SPLITS:
        split_dir = path / split
        if not split_dir.exists():
            continue
        for class_dir in split_dir.iterdir():
            if not class_dir.is_dir():
                continue
            target_cls = map_class(class_dir.name)
            if target_cls is None:
                unmapped_classes_seen.add((name, class_dir.name))
                continue

            imgs = [p for p in class_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            for img_path in imgs:
                out_name = f"{name}_{split}_{img_path.stem}.jpg"
                out_path = MERGED_DIR / split / "images" / out_name
                try:
                    Image.open(img_path).convert("RGB").save(out_path, "JPEG", quality=95)
                except Exception:
                    continue
                manifest_rows.append({
                    "split": split, "filename": out_name,
                    "class_id": CLASS_TO_ID[target_cls], "has_mask": 0, "source": name,
                })
            counts[split][target_cls] = counts[split].get(target_cls, 0) + len(imgs)
    return counts

## Step 3 — Ingestion : datasets `coco` (classification + **vrais masques de segmentation**)

Différence clé avec la v1 : les polygones COCO sont **rasterisés en un
masque pixel-par-pixel à la taille de l'image complète**, au lieu de
servir uniquement à recadrer un patch. Chaque image source peut porter
plusieurs annotations (plusieurs instances/classes) : elles sont toutes
dessinées sur le même masque (les polygones qui se chevauchent, la
dernière annotation dessinée l'emporte -- simplification documentée).

Le **label de classification** de l'image est déterminé par la classe
dont les instances couvrent la plus grande surface totale (vote par aire,
pas juste par nombre d'instances) -- une image avec un grand corail mort
et un petit corail sain à l'arrière-plan est labellisée `Dead_Coral`.

In [4]:
def rasterize_mask(img_size, annotations):
    """
    img_size: (W, H)
    annotations: liste de (target_cls: str, segmentation: list|None, bbox: list)
    Retourne (mask: np.uint8 array HxW avec CLASS_TO_ID, area_by_class: dict)
    """
    mask_img = Image.new("L", img_size, 0)
    draw = ImageDraw.Draw(mask_img)
    area_by_class = defaultdict(int)

    for target_cls, seg, bbox in annotations:
        class_id = CLASS_TO_ID[target_cls]
        if seg and isinstance(seg, list) and len(seg) > 0 and isinstance(seg[0], list):
            poly = [(seg[0][i], seg[0][i + 1]) for i in range(0, len(seg[0]), 2)]
            draw.polygon(poly, outline=class_id, fill=class_id)
        elif bbox:
            x, y, w, h = bbox
            draw.rectangle([x, y, x + w, y + h], outline=class_id, fill=class_id)
        else:
            continue

    mask_arr = np.array(mask_img, dtype=np.uint8)
    for class_id in np.unique(mask_arr):
        if class_id == 0:
            continue
        area_by_class[int(class_id)] = int((mask_arr == class_id).sum())
    return mask_arr, area_by_class


def ingest_coco_dataset(name, path):
    counts = {"train": {}, "valid": {}, "test": {}}
    for split in SPLITS:
        ann_path = path / split / "_annotations.coco.json"
        if not ann_path.exists():
            continue
        with open(ann_path) as f:
            coco = json.load(f)

        cat_map = {c["id"]: c["name"] for c in coco["categories"]}
        img_map = {img["id"]: img for img in coco["images"]}

        # regroupe les annotations par image
        anns_by_image = defaultdict(list)
        for ann in coco["annotations"]:
            raw_cls = cat_map.get(ann["category_id"])
            target_cls = map_class(raw_cls) if raw_cls else None
            if target_cls is None:
                if raw_cls:
                    unmapped_classes_seen.add((name, raw_cls))
                continue
            anns_by_image[ann["image_id"]].append(
                (target_cls, ann.get("segmentation"), ann.get("bbox"))
            )

        for image_id, anns in anns_by_image.items():
            img_info = img_map.get(image_id)
            if img_info is None:
                continue
            img_path = path / split / img_info["file_name"]
            if not img_path.exists():
                continue

            try:
                img = Image.open(img_path).convert("RGB")
            except Exception:
                continue

            mask_arr, area_by_class = rasterize_mask(img.size, anns)
            if not area_by_class:
                continue

            # label de classification = classe de plus grande surface totale
            dominant_class_id = max(area_by_class, key=area_by_class.get)
            dominant_class_name = CLASSES[dominant_class_id - 1]

            out_stem = f"{name}_{split}_{image_id}"
            img.save(MERGED_DIR / split / "images" / f"{out_stem}.jpg", "JPEG", quality=95)
            Image.fromarray(mask_arr, mode="L").save(MERGED_DIR / split / "masks" / f"{out_stem}.png")

            manifest_rows.append({
                "split": split, "filename": f"{out_stem}.jpg",
                "class_id": dominant_class_id, "has_mask": 1, "source": name,
            })
            counts[split][dominant_class_name] = counts[split].get(dominant_class_name, 0) + 1
    return counts

## Step 4 — Lancer le merge sur les 5 sources

In [5]:
all_counts = {}
for name, cfg in DATASETS.items():
    if not cfg["path"].exists():
        print(f"Skipping {name}: path not found")
        continue
    print(f"Ingesting {name} ({cfg['type']}) ...")
    if cfg["type"] == "folder":
        c = ingest_folder_dataset(name, cfg["path"])
    else:
        c = ingest_coco_dataset(name, cfg["path"])
    all_counts[name] = c
    print(f"  -> {c}")

print("\n" + "=" * 60)
print("CATÉGORIES BRUTES NON MAPPÉES (ajoutez-les à CLASS_MAP si besoin) :")
for ds, cls in sorted(unmapped_classes_seen):
    print(f"  dataset={ds:30s} raw_class={cls}")

manifest_df = pd.DataFrame(manifest_rows)
print(f"\nTotal avant déduplication : {len(manifest_df)} images "
      f"({manifest_df['has_mask'].sum()} avec masque réel)")

Ingesting bleach_detection (folder) ...
  -> {'train': {'Bleached_Coral': 4980, 'Healthy': 4682}, 'valid': {'Bleached_Coral': 240, 'Healthy': 223}, 'test': {'Bleached_Coral': 135, 'Healthy': 122}}
Ingesting coral_bleaching_fish_reefs (coco) ...
  -> {'train': {'Healthy': 2406, 'Bleached_Coral': 2625}, 'valid': {'Bleached_Coral': 249, 'Healthy': 225}, 'test': {'Bleached_Coral': 139, 'Healthy': 125}}
Ingesting graduation_design (coco) ...
  -> {'train': {'Healthy': 587, 'Bleached_Coral': 436, 'Dead_Coral': 785}, 'valid': {'Dead_Coral': 226, 'Healthy': 158, 'Bleached_Coral': 138}, 'test': {'Healthy': 83, 'Dead_Coral': 108, 'Bleached_Coral': 63}}
Ingesting marjan_balance (coco) ...
  -> {'train': {'Bleached_Coral': 1557, 'Dead_Coral': 2876, 'Healthy': 1975}, 'valid': {'Bleached_Coral': 70, 'Dead_Coral': 104, 'Healthy': 88}, 'test': {'Healthy': 92, 'Dead_Coral': 120, 'Bleached_Coral': 64}}
Ingesting surge (coco) ...
  -> {'train': {'Bleached_Coral': 2901, 'Healthy': 300}, 'valid': {'Healthy

## Step 5 — Déduplication GLOBALE (correction du problème 1)

Calcule le hash SHA256 de **toutes** les images fusionnées, tous splits
confondus. Pour chaque groupe de doublons, ne garde qu'**une seule**
copie -- priorité `train` > `valid` > `test`, pour maximiser les données
d'entraînement sans jamais laisser une image dupliquée contaminer
`valid`/`test`. Les fichiers (image + masque éventuel) et les lignes de
manifeste des doublons supprimés sont retirés.

In [6]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


split_priority = {"train": 0, "valid": 1, "test": 2}

print("Calcul des hashs SHA256 sur toutes les images fusionnées...")
manifest_df["hash"] = [
    file_sha256(MERGED_DIR / row.split / "images" / row.filename)
    for row in tqdm(manifest_df.itertuples(), total=len(manifest_df))
]

manifest_df["split_rank"] = manifest_df["split"].map(split_priority)
manifest_df = manifest_df.sort_values("split_rank")

before = len(manifest_df)
# garde la première occurrence (donc train en priorité) de chaque hash
dedup_df = manifest_df.drop_duplicates(subset="hash", keep="first").copy()
removed_df = manifest_df[~manifest_df.index.isin(dedup_df.index)]

print(f"Images avant déduplication : {before}")
print(f"Doublons supprimés         : {len(removed_df)}")
print(f"Images après déduplication : {len(dedup_df)}")

# supprime physiquement les fichiers des doublons retirés
for row in removed_df.itertuples():
    img_path = MERGED_DIR / row.split / "images" / row.filename
    mask_path = MERGED_DIR / row.split / "masks" / (Path(row.filename).stem + ".png")
    img_path.unlink(missing_ok=True)
    if row.has_mask:
        mask_path.unlink(missing_ok=True)

manifest_df = dedup_df.drop(columns=["hash", "split_rank"]).reset_index(drop=True)

# vérification : plus aucun hash ne doit apparaître dans 2 splits différents
verify_hashes = {
    file_sha256(MERGED_DIR / row.split / "images" / row.filename): row.split
    for row in manifest_df.itertuples()
}
print(f"\nVérification : {len(verify_hashes)} hashs uniques pour "
      f"{len(manifest_df)} images -> {'OK, aucune fuite restante' if len(verify_hashes)==len(manifest_df) else 'ATTENTION incohérence'}")

Calcul des hashs SHA256 sur toutes les images fusionnées...


100%|██████████| 30255/30255 [06:12<00:00, 81.20it/s] 


Images avant déduplication : 30255
Doublons supprimés         : 906
Images après déduplication : 29349

Vérification : 29349 hashs uniques pour 29349 images -> OK, aucune fuite restante


## Step 6 — Distribution après déduplication (avant augmentation)

In [7]:
def summarize(df):
    return (df.groupby(["split", "class_id"]).size()
              .unstack(fill_value=0)
              .rename(columns={v: k for k, v in CLASS_TO_ID.items()})
              .reindex(SPLITS))

print("Distribution par classe (après déduplication, avant augmentation) :")
print(summarize(manifest_df))
print()
print("Répartition has_mask par split :")
print(manifest_df.groupby("split")["has_mask"].agg(["sum", "count"]))

Distribution par classe (après déduplication, avant augmentation) :
class_id  Healthy  Dead_Coral  Bleached_Coral
split                                        
train        9754        3505           12197
valid         740         290            1460
test          447         185             771

Répartition has_mask par split :
         sum  count
split              
test    1153   1403
train  16022  25456
valid   2027   2490


## Step 7 — Augmentation class-aware (train uniquement)

Même logique que la v1 (rotation, flip, luminosité/contraste, bruit
gaussien) pour équilibrer les classes minoritaires du train, **mais**
la transformation géométrique (rotation/flip) est désormais appliquée
**identiquement à l'image et à son masque** quand celui-ci existe
(interpolation NEAREST pour le masque -- jamais d'interpolation qui
mélangerait des indices de classe). Les perturbations photométriques
(luminosité, bruit) ne s'appliquent qu'à l'image, jamais au masque.

In [8]:
from PIL import ImageEnhance

TARGET_PER_CLASS_TRAIN = manifest_df[manifest_df.split == "train"]["class_id"].value_counts().max()

def augment_pair(img, mask):
    angle = random.uniform(-30, 30)
    img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    if mask is not None:
        mask = mask.rotate(angle, resample=Image.NEAREST, fillcolor=0)

    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if mask is not None:
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        if mask is not None:
            mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))

    arr = np.array(img).astype(np.int16)
    noise = np.random.normal(0, 8, arr.shape).astype(np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)

    return img, mask


train_df = manifest_df[manifest_df.split == "train"]
new_rows = []
for class_id in CLASS_TO_ID.values():
    class_rows = train_df[train_df.class_id == class_id]
    n_needed = TARGET_PER_CLASS_TRAIN - len(class_rows)
    if n_needed <= 0:
        continue
    if len(class_rows) == 0:
        # Classe absente du train apres dedup : impossible d augmenter a
        # partir de rien -- avertit plutot que de planter (bug trouve en
        # testant : class_rows.sample() sur un DataFrame vide leve une
        # ValueError).
        print(f"ATTENTION : classe {CLASSES[class_id-1]} absente du train "
              f"apres deduplication -- aucune augmentation possible, "
              f"verifier vos sources ou CLASS_MAP.")
        continue
    print(f"Classe {CLASSES[class_id-1]}: +{n_needed} images augmentées")

    for i in range(n_needed):
        src_row = class_rows.sample(1, random_state=i).iloc[0]
        img = Image.open(MERGED_DIR / "train" / "images" / src_row.filename).convert("RGB")
        mask = None
        if src_row.has_mask:
            mask = Image.open(MERGED_DIR / "train" / "masks" / (Path(src_row.filename).stem + ".png"))

        aug_img, aug_mask = augment_pair(img, mask)

        out_stem = f"aug_{class_id}_{i}_{Path(src_row.filename).stem}"
        aug_img.save(MERGED_DIR / "train" / "images" / f"{out_stem}.jpg", "JPEG", quality=95)
        if aug_mask is not None:
            aug_mask.save(MERGED_DIR / "train" / "masks" / f"{out_stem}.png")

        new_rows.append({
            "split": "train", "filename": f"{out_stem}.jpg",
            "class_id": class_id, "has_mask": int(src_row.has_mask), "source": f"aug_{src_row.source}",
        })

manifest_df = pd.concat([manifest_df, pd.DataFrame(new_rows)], ignore_index=True)
print(f"\nTotal après augmentation : {len(manifest_df)} images")

Classe Healthy: +2443 images augmentées
Classe Dead_Coral: +8692 images augmentées

Total après augmentation : 40484 images


## Step 8 — Distribution finale (post-augmentation)

In [9]:
print("Distribution finale par classe :")
print(summarize(manifest_df))
print()
print("Répartition has_mask par split (proportion de vrais masques) :")
mask_summary = manifest_df.groupby("split")["has_mask"].agg(["sum", "count"])
mask_summary["pct_with_mask"] = (100 * mask_summary["sum"] / mask_summary["count"]).round(1)
print(mask_summary)

Distribution finale par classe :
class_id  Healthy  Dead_Coral  Bleached_Coral
split                                        
train       12197       12197           12197
valid         740         290            1460
test          447         185             771

Répartition has_mask par split (proportion de vrais masques) :
         sum  count  pct_with_mask
split                             
test    1153   1403           82.2
train  26011  36591           71.1
valid   2027   2490           81.4


## Step 9 — Poids de classe (pour la perte de classification pondérée)

In [10]:
train_counts = manifest_df[manifest_df.split == "train"]["class_id"].value_counts().sort_index()
total_train = train_counts.sum()
num_classes = len(CLASSES)

class_weights = {
    CLASSES[cid - 1]: total_train / (num_classes * count)
    for cid, count in train_counts.items()
}
print("Poids de classe (Eq. 1 de l'article) :")
for cls, w in class_weights.items():
    print(f"  {cls:15s}: {w:.6f}")

Poids de classe (Eq. 1 de l'article) :
  Healthy        : 1.000000
  Dead_Coral     : 1.000000
  Bleached_Coral : 1.000000


## Step 10 — Écriture du manifeste final (`labels.csv` par split)

Format directement consommable par `datasets/coral_dataset.py::CoralDataset`
du pipeline UWOC-SemCom : `filename,class_id,has_mask`.
`has_mask=0` signifie qu'aucun masque réel n'existe pour cette image
(source `bleach_detection`, classification pure) -- **le pipeline doit
ignorer la perte de segmentation sur ces échantillons**, pas leur
attribuer un masque vide qui polluerait l'entraînement de la tête de
segmentation (voir note finale).

In [11]:
for split in SPLITS:
    split_df = manifest_df[manifest_df.split == split][["filename", "class_id", "has_mask"]]
    split_df.to_csv(MERGED_DIR / split / "labels.csv", index=False)
    print(f"{split}: {len(split_df)} lignes écrites -> {MERGED_DIR / split / 'labels.csv'}")

# petit fichier récapitulatif JSON, pratique pour audit/inspection ultérieure
summary = {
    "class_to_id": CLASS_TO_ID,
    "split_counts": manifest_df.groupby("split").size().to_dict(),
    "class_counts_by_split": {
        split: manifest_df[manifest_df.split == split]["class_id"].value_counts().to_dict()
        for split in SPLITS
    },
    "has_mask_counts_by_split": {
        split: manifest_df[manifest_df.split == split]["has_mask"].value_counts().to_dict()
        for split in SPLITS
    },
    "class_weights": class_weights,
}
with open(MERGED_DIR / "merge_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("\nRésumé écrit ->", MERGED_DIR / "merge_summary.json")

train: 36591 lignes écrites -> merged_dataset\train\labels.csv
valid: 2490 lignes écrites -> merged_dataset\valid\labels.csv
test: 1403 lignes écrites -> merged_dataset\test\labels.csv

Résumé écrit -> merged_dataset\merge_summary.json


## Step 11 — Vérification finale anti-fuite

Re-vérifie, après augmentation, qu'aucune image augmentée n'a par
inadvertance recréé une fuite inter-split (impossible en théorie puisque
l'augmentation ne touche que le train, mais vérification de sécurité).

In [12]:
final_hashes = defaultdict(set)
for row in tqdm(manifest_df.itertuples(), total=len(manifest_df), desc="Vérification finale"):
    h = file_sha256(MERGED_DIR / row.split / "images" / row.filename)
    final_hashes[h].add(row.split)

leaking = {h: splits for h, splits in final_hashes.items() if len(splits) > 1}
if leaking:
    print(f"ATTENTION : {len(leaking)} hash(s) encore présents dans plusieurs splits !")
    for h, splits in list(leaking.items())[:5]:
        print(f"  {h[:12]}... -> {splits}")
else:
    print("OK : aucune fuite de données entre splits après augmentation.")

Vérification finale: 100%|██████████| 40484/40484 [02:05<00:00, 323.79it/s]

OK : aucune fuite de données entre splits après augmentation.


## Notes pour brancher sur `datasets/coral_dataset.py::CoralDataset`

Le manifeste `labels.csv` inclut désormais une colonne `has_mask` que la
version actuelle de `CoralDataset` (pipeline UWOC-SemCom) ne lit pas
encore. Deux options pour finaliser le branchement :

1. **Recommandé** — étendre `CoralDataset.__getitem__` pour lire
   `has_mask` depuis `labels.csv` et renvoyer ce flag dans le dict de
   sortie (`{'image', 'mask', 'label', 'has_mask', 'filename'}`). Dans
   `losses/latent_align.py::compute_phase4_loss`, ne calculer la perte de
   segmentation (CE + Dice) que sur les échantillons du batch où
   `has_mask == 1` (masquer/filtrer les autres) -- pour ne jamais laisser
   les ~{pct}% d'images `bleach_detection` (sans masque réel) polluer
   l'entraînement de `seg_head` avec un masque vide interprété comme
   "tout est fond".
2. **Alternative plus simple** — n'entraîner `seg_head` que sur un
   sous-ensemble filtré (`has_mask == 1`) via un second DataLoader dédié à
   la segmentation, et garder le DataLoader complet (toutes images) pour
   `cls_head` uniquement en Phase 4.

Dans les deux cas, `cls_head` peut s'entraîner sur l'intégralité du
dataset fusionné (100% des images ont un label de classification, qu'un
masque existe ou non).